# Segmentação com YOLO

Quando começamos a trabalhar com visão computacional, classificamos imagens de acordo com um determinado objeto ou padrão que há nela. Então começamos a evoluir, pensando em determinar em que ponto da imagem o objeto estava. Agora, além de saber onde o objeto está, queremos saber o contorno exato do objeto.

## Estrutura dos dados

A estrutura dos dados quando trabalhamos com segmentação é exatamente igual a quando trabalhamos com detecção. Isso não mudará independente de que função especial do YOLO estivermos aplicando. Mas, uma coisa muda, e é o conteúdo nos arquivos de texto.

Para contornar uma forma de um objeto na imagem, precisamos de muito mais do que apenas quatro pontos, como é típico na detecção que desenha apenas um retângulo. Aqui, será desenhado um polígono irregular que terá vários pontos principais conectados sequencialmente para formar o contorno.

Observe o esquema do arquivo abaixo. Primeiro, temos a classe. Depois, temos vários pares de coordenadas X e Y. Cada par representa um ponto na imagem.

```text
<classe> <x1> <y1> <x2> <y2> ... <xn> <yn>
```

Abaixo, um exemplo com valores reais de um dataset de objetos de trânsito. As coordenadas X e Y dos pontos do polígono sempre tem de estar normalizadas. As coordenadas X serão divididas pela largura das imagens e as coordenadas Y serão divididas pela altura das imagens. Dessa forma, os pontos serão sempre representados por números entre 0 e 1.

```text
10 0.222075 0.207265 0.222075 0.214744 0.229317 0.211538
7 0.222880 0.377137 0.222880 0.396368 0.272766 0.395299 0.273571 0.373932
8 0.276790 0.365385 0.277594 0.419872 0.280813 0.417735 0.280008 0.369658
```

## Arquivo de Configuração dos Dados

O arquivo de configuração também se mantém essencialmente igual, definindo o caminho base para o diretório que contém os dados, a definição de todas as classes, do número de classes e dos subconjuntos de dados para treino, validação e teste. Para este projeto, além das configurações padrão dos dados, também temos uma seção de configurações do roboflow, pois os dados utilizados no projeto pertencem à terceiros.

```yaml
path: ./data
train: train/images
val: valid/images
test: test/images

nc: 2
names: ['Fogo', 'Fumaça']

roboflow:
  workspace: jose-anselmo-ferreira-matos
  project: fire-smoke-segmentation-3ym8x-2wv2q
  version: 1
  license: CC BY 4.0
  url: https://app.roboflow.com/jose-anselmo-ferreira-matos/fire-smoke-segmentation-3ym8x-2wv2q/1
```

## Métricas de Avaliação

Os modelos de segmentação, além de identificar a classe do objeto, também desenham a caixa retangular que o contém e um polígono irregular de contorno. Portanto, avaliam também esses três aspectos. O cálculo de precisão para a predição de classes e para a sobreposição das caixas retangulares continua exatamente igual. A novidade é a pontuação IoU para a máscara. O que chamamos de máscara é justamente o polígono que envolve o objeto. Para a máscara, a pontuação IoU é calculada dividindo a quantidade de pixels que os dois polígonos contém em comum (interseção) pela quantidade de pixels que os dois polígonos contém em comum ou separadamente (união).

### Pontuações mAP

As pontuações mAP para a máscara funcionam exatamente como para as caixas. A diferença é apenas como medimos o sucesso da pontuação. Para um mAP@50-95, valores acima de 0.35 a 0.45 já são considerados adequados para aplicações de nível comercial avançado.

## Escolha do Modelo

Na tabela abaixo, descrevemos as variantes dos modelos YOLO para segmentação. Os nomes dos pesos mudam apenas adicionando um sufixo "-seg" ao final do nome, antes da extensão do arquivo. Também variam, é claro, no número de parâmetros, tamanho e latência.

| Variante | Pesos | Parâmetros | Tamanho | Latência | Cenário Ideal |
| -------- | ----- | ---------- | ------- | -------- | ------------- |
| Nano (n) | yolov8n-seg.pt, yolo11n-seg.pt | ~3.4M | ~7 MB | ~2.0 ms | Dispositivos de borda como Raspberry Pi e celulares. Viável para rodar em CPU ou para prototipagem básica |
| Small (s) | yolov8s-seg.pt, yolo11s-seg.pt | ~11.8M | ~23 MB | ~3.5 ms | Equilíbrio entre acurácia e performance |
| Medium (m) | yolov8m-seg.pt, yolo11m-seg.pt | ~27.3M | ~52 MB | ~6.5 ms | Tarefas complexas com dezenas de classes e sobreposição visual |
| Large (l) | yolov8l-seg.pt, yolo11l-seg.pt | ~46.0M | ~88 MB | ~10.0 ms | Quando a GPU de inferência  potente e a precisão é prioritária |
| XLarge (x) | yolov8x-seg.pt, yolo11x-seg.pt | ~71.8M | ~137 MB | ~15.0 ms | Competições acadêmicas e benchmarks de pesquisa |

## Transfer Learning

Se você ainda se lembra de como o transfer learning era realizado nos modelos simples de YOLO, saiba: é exatamente igual para modelos de segmentação. A única diferença aqui é que instanciamos o modelo com pesos diferentes.

In [ ]:
from ultralytics import YOLO

from segmentacao.config import (
    BATCH_SIZE,
    DATASET_YAML,
    DEVICE,
    IMAGE_SIZE,
    NUM_EPOCHS,
    NUM_WORKERS,
    OPTIMIZER,
    OUTPUT_DIR,
    PATIENCE,
    SAVE_PERIOD,
)


def train():
    model = YOLO("yolov8s-seg.pt")

    results = model.train(
        data=DATASET_YAML,         # Caminho até o arquivo de configuração dos dados
        epochs=NUM_EPOCHS,         # Número de épocas (60)
        batch=BATCH_SIZE,          # Tamanho do lote (8)
        imgsz=IMAGE_SIZE,          # Tamanho de redimensionamento das imagens (640)
        device=DEVICE,             # Dispositivo para treino (0 = cuda)
        workers=NUM_WORKERS,       # Número de subprocessos para carregar lotes (4)
        optimizer=OPTIMIZER,       # Algoritmo de otimização ("auto" = SGD + AdamW)
        patience=PATIENCE,         # Épocas para early stopping (15)
        save=True,                 # Salvar o modelo
        save_period=SAVE_PERIOD,   # Salvar o modelo a cada X épocas (10)
        project=OUTPUT_DIR,        # Diretório de saída dos artefatos
        name="training",           # Subpasta no diretório de saída
        verbose=True
    )

    print("Treinamento finalizado com sucesso!")
    print(f"Melhores pesos salvos em: {results.save_dir}/weights/best.pt")

if __name__ == "__main__":
    train()

## Artefatos de Validação

Dentro da pasta de execução de um treinamento, existe a pasta weights que carrega arquivos conforme o esquema abaixo.

```shell
├── ...
│   └── weights
│       ├── best.pt          # Pesos da época com o maior mAP50-95
│       ├── epoch0.pt        # Checkpoint a cada X épocas
│       ├── epoch10.pt       # Checkpoint a cada X épocas
│       ├── epoch20.pt       # Checkpoint a cada X épocas
│       ├── epoch30.pt       # Checkpoint a cada X épocas
│       ├── epoch40.pt       # Checkpoint a cada X épocas
│       ├── epoch50.pt       # Checkpoint a cada X épocas
│       └── last.pt          # Pesos da última época do treinamento
└── ...
```

Além deles, há arquivos visuais que nos ajudam a entender e visualizar métricas do modelo treinado.

```shell
├── ...
│   ├── weights
│   ├── val_batch*_pred.jpg                 # Imagens da validação com caixas e máscaras desenhadas
│   ├── BoxF1_curve.png                     # Curva entre confiança e acurácia das caixas
│   ├── MaskF1_curve.png                     # Curva entre confiança e acurácia das máscaras
│   ├── confusion_matrix_normalized.png     # Matriz de confusão normalizada
│   └── results.png                         # Gráficos de progressão do treinamento
└── ...
```

### Imagens de Validação com Caixas e Máscaras Desenhadas

A amostra de imagem abaixo contém todas as imagens de um lote. Essas imagens foram processadas pelo modelo e foram devolvidas com as caixas, classes preditas e máscaras desenhados. Esses arquivos servem para verificarmos visualmente o que o modelo foi capaz de fazer durante a etapa de validação.

![Figura 9](assets/images/figura_09.jpg)

### Curva de Confiança e Acurácia das Caixas

Esta imagem é um gráfico de linha que revela a relação entre a confiança esperada e acurácia para as caixas. Para uma confiança esperada de 0.5, por exemplo, o modelo alcançou uma pontuação F1 de quase 0.80 (80%).

![Figura 10](assets/images/figura_10.png)

### Curva de Confiança e Acurácia das Caixas

Esta imagem é um gráfico de linha que revela a relação entre a confiança esperada e acurácia para as máscaras. Para uma confiança esperada de 0.7, por exemplo, o modelo alcançou uma pontuação F1 de quase 0.70 (70%).

![Figura 11](assets/images/figura_11.png)

### Matriz de Confusão Normalizada

A matriz de confusão normalizada é o que nos dirá como se distribuíram as classificações do modelo. No eixo X há os valores reais e no eixo Y os valores preditos. Neste exemplo, o modelo acertou 95% dos objetos classificados como "Fogo" e 94% dos objetos classificamos como "Fumaça".

![Figura 12](assets/images/figura_12.png)

### Progressão do Treinamento

Os gráficos de progressão do treinamento desenham uma linha através de todas as épocas do treinamento, revelando como uma determinada métrica evoluiu durante a sessão completa. Idealmente, as métricas box_loss, seg_loss, cls_loss e dfl_loss tem de andar grosseiramente juntas. Imagine que ao mesmo tempo que a perda de treinamento diminui, a perda da validação aumenta. Isso pode indicar que o modelo não está conseguindo extrair sinais valiosos da imagem, e sim "decorando" o conjunto de dados. Há também outros gráficos que mostram como evoluiram as métricas de acurácia, recall, mAP50 e mAP50-95.

![Figura 13](assets/images/figura_13.png)

## Inferência

Para realizar inferências com modelos de segmentação, assim como com modelos simples, utilizamos o método predict. Ele receberá parâmetros para definir a origem dos dados, dispositivo usado, configurações de salvamento dos resultados, etc. Assim como nos outros modelos, podemos utilizar a webcam do computador como origem ou até mesmo uma câmera de segurança via protocolo RSTP (Rapid Spanning Tree Protocol).

In [ ]:
from ultralytics import YOLO

from segmentacao.config import DEVICE, INPUT_DIR

model = YOLO("yolov8s-seg.pt")

model.predict(
    source=INPUT_DIR,
    device=DEVICE,
    save=True,
)

Neste exemplo, estamos utilizando os pesos de um modelo treinado para detectar fogo e fumaça com as máscaras de segmentação. Eis o código que utilizamos no arquivo de inferência. Após a predição, processamos os resultados extraindo o id da classe de cada caixa, nome da classe, confiança, coordenadas da bounding box e exibimos no terminal.

In [ ]:
from detection.config import OUTPUT_DIR
from detection.config import INPUT_DIR
from detection.config import WEIGHTS_PATH

from ultralytics import YOLO

model = YOLO(WEIGHTS_PATH)

results = model.predict(
    source=INPUT_DIR,
    verbose=False,
    conf=0.5,
    save=True, 
    project=OUTPUT_DIR,
    name="infer",
    exist_ok=True
)

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0])
        class_name = result.names[class_id]

        confidence = float(box.conf[0])

        x1, y1, x2, y2 = box.xyxy[0].int().tolist()

        print(
            f"""
-> {class_name}
   Confiança: {confidence:.2%}
   Dimensões: {x2 - x1}x{y2 - y1}
   Coordenadas: {x1, y1, x2, y2}
"""
        )


**Imagem demonstrativa original**

![Figura 7](assets/images/figura_07.jpg)

**Demonstração de resultados de predição**

![Figura 8](assets/images/figura_08.jpg)

**Saída no terminal**

```shell
-> Fogo
   Confiança: 83.47%
   Dimensões: 294x207
   Coordenadas: (1153, 479, 1447, 686)


-> Fumaça
   Confiança: 67.16%
   Dimensões: 1537x714
   Coordenadas: (9, 7, 1546, 721)
```

### Calculando Área de Segmentação

Além de conseguirmos extrair as classes, caixas e confianças, conseguimos também extrair as máscaras. A propriedade masks de cada resultado terá a máscara para a respectiva caixa. Utilizando essa propriedade, conseguimos acessar um tensor numpy que terá números normalizados entre 0 e 1. Para cada valor do tensor, aquele valor representa a certeza de que o pixel reference ao valor faz parte da máscara identificada para o objeto. 

Podemos utilizar esse tensor para muitas coisas. Uma delas é calcular o quanto da imagem o objeto identificado ocupa. Para isso, redimensionamos o tensor para coincidir com as dimensões da imagem, filtramos os valores que superam a confiança mínima e então contamos quantos deles existem. O resultado será o número de pixels da imagem que compõe a máscara. Dividindo esse número pelo total de pixels da imagem, teremos uma porcentagem do quanto da imagem o objeto identificado ocupa.

In [ ]:
import numpy as np
import cv2


# ...

for result in results:
    original_height, original_width = result.orig_shape

    for index, box in enumerate(result.boxes):
        # ...

        mask = result.masks[index].data[0].cpu().numpy()
        mask_scaled = cv2.resize(mask, (original_width, original_height))
        mask_binary = (mask_scaled >= 0.5).astype(int)

        total_image_pixels = original_width * original_height

        coverage = np.count_nonzero(mask_binary) / total_image_pixels

        print(
            f"""
-> {class_name}
   Confiança: {confidence:.2%}
   Dimensões: {x2 - x1}x{y2 - y1}
   Coordenadas: {x1, y1, x2, y2}
   Cobertura: {coverage:.2%}
"""
        )


```shell
-> Fogo
   Confiança: 79.92%
   Dimensões: 296x207
   Coordenadas: (1155, 480, 1451, 687)
   Cobertura: 1.71%


-> Fumaça
   Confiança: 68.27%
   Dimensões: 1525x716
   Coordenadas: (4, 3, 1529, 719)
   Cobertura: 37.36%
```

## Exportando o Modelo

Da mesma forma que exportamos um modelo comum de detecção, podemos exportar um modelo de segmentação para o formato ONNX. Permitindo utilizar o modelo em dispositivos com capacidade computacional menor. Ex.: com cpu mais fraca, com menos memória RAM, sem placa de vídeo, etc.

In [ ]:
from ultralytics import YOLO

from detection.config import IMAGE_SIZE, WEIGHTS_PATH

model = YOLO(WEIGHTS_PATH)

onnx_path = model.export(
    format="onnx",
    imgsz=IMAGE_SIZE,
    dynamic=True,
    simplify=True
)

print(f"Modelo exportado para: {onnx_path}")